In [2]:
from collections import defaultdict
import pandas as pd

In [7]:
# dict creation with mgi-mp from DO db
mgi_mp_DO = defaultdict(set)
f = open("../databases/MGI_Geno_DiseaseDO.rpt.txt", "r")
for line in f:
    line_data = line.rstrip().split("\t")
    mp = line_data[4]
    mgi = line_data[6]
    mgi_mp_DO[mgi].add(mp)
f.close()

In [9]:
len(mgi_mp_DO) # 1550 genes from DO db

1550

In [17]:
df = pd.read_csv("db_1_hum_mouse_genes.csv", header=0, sep='\t', index_col=0)
mgi_mp_bd_1 = defaultdict(list)
for elem in df.itertuples():
    gene_hum = elem[1]
    gene = elem[4]
    phenos = elem[5]
    mgi_mp_bd_1[gene].extend(str(phenos).split(", "))

In [19]:
mgi_mp_set_bd_1 = {k: set(v) for k, v in db_1_gene_to_phenotypes.items()}
print(len(mgi_mp_set_bd_1))

20131


In [20]:
gene_to_phenos_merged = {}
keys_in_both = set(mgi_mp_set_bd_1.keys()).union(mgi_mp_DO.keys())
print(len(keys_in_both))

20222


In [21]:
f = open("mgi_mp.tsv", "w")
for key in keys_in_both:
    from_db1 = mgi_mp_set_bd_1.get(key, set())
    from_do = mgi_mp_DO.get(key, set())
    in_both = from_db1.union(from_do)
    if len(in_both) > 0:
        both_stringed = ",".join(in_both)
    else:
        both_stringed = "None"
    f.write(f"{key}\t{both_stringed}\n")
f.close()

In [8]:
db_1 = pd.read_csv("db_1_hum_mouse_genes.csv", delimiter='\t')
db_1 = db_1.drop("MP", axis=1)
db_mouse_mp = pd.read_csv("mgi_mp.tsv", delimiter='\t', header=None).rename(columns={0: "MGI", 1: "MP"})

In [11]:
db_hum_mouse_mp = pd.merge(db_1, db_mouse_mp, how="left", on="MGI")
db_mouse_mp.to_csv("general_bd_with_mp.tsv", sep='\t')